Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-gemma/gbench/blob/main/examples/notebooks/02_workloads_campaigns_and_concurrency.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench workload campaigns, concurrency scaling, and prompt geometry

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook explores how to evaluate foundation model performance under realistic production workloads using `gbench`. You will test concurrency scaling by sweeping batch sizes, configure custom prompt geometries, and run predefined performance campaigns (`chat-like`, `agentic`, `decode-heavy`) against a local Ollama serving engine.

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 8192 tokens.
2. Execute concurrency sweeps (`--batch-sizes 1 4 8 16`) to measure latency and throughput scaling curves.
3. Test custom input and output prompt token lengths (`--input-lengths` and `--output-lengths`).
4. Execute and compare standardized production workload campaigns (`chat-like`, `agentic`, `decode-heavy`).
5. Terminate the local Ollama session and reclaim system memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-gemma/gbench)
* [Ollama documentation](https://github.com/ollama/ollama)
* [ShareGPT V3 dataset](https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered)

## 1. Environment setup and installation

We clone the `gbench` repository from GitHub, change directory into the project root (`%cd gbench`), and install the package in editable mode (`pip install -e .`). This builds and links the `gbench` CLI executable without installing unnecessary development linters.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-gemma/gbench.git
    %cd -q /content/gbench
else:
    if not Path("pyproject.toml").is_file() and not Path("gbench").is_dir():
        if not Path("gbench").is_dir():
            !git clone https://github.com/google-gemma/gbench.git
        %cd gbench

%pip install -e . -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available presets
!gbench --list presets

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus tokenizers/datasets from the Hugging Face Hub with `huggingface_hub` — some are **gated** — so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model/dataset page you use.
2. On **Colab**: click the **🔑 key icon (Secrets)** in the left sidebar → **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector)
# and tokenizers/datasets from the Hugging Face Hub via huggingface_hub, which
# authenticates with it (resumable, higher rate limits, and access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless BOTH the binary and its llama-server runner are present.
# A binary-only partial install fails every request with "llama-server binary not
# found", so checking only for the binary would skip the repair.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 8192`**: Context window of 8192 tokens.
* **`SYSTEM prompt`**: System instruction defining Gemma 4 AI assistant capabilities.

We download here rather than letting `ollama create` pull `hf.co/…` itself: Ollama's puller is anonymous (it can't use `HF_TOKEN`) and can stall on the HF CDN. The two-`FROM` import (main GGUF + `mmproj`) keeps the model's **vision** capability.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) via huggingface_hub, which
# authenticates with HF_TOKEN - Ollama's own hf.co puller is anonymous and can stall
# on the HF CDN. Build the model FROM the local files: a two-FROM import (main +
# mmproj) keeps gemma-4's vision capability (verified with `ollama show`). realpath
# resolves the HF cache symlink so `ollama create` reads the actual file.
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 8192',
          'SYSTEM "You are a helpful Gemma 4 AI assistant with reasoning, vision, and tool calling capabilities."']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk — no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Concurrency scaling sweeps

To evaluate how latency and throughput scale under increasing client concurrency, we pass `--batch-sizes 1 4 8 16`. gbench runs the **serving** pillar once per batch size (a 4-point sweep here) and, by default, follows it with an **arrival-rate stress test** that ramps request QPS to find the sustainable-throughput knee. In other words this is a full serving-plus-stress campaign, not a quick serving-only pass. Use `--serving-only` if you want to skip the stress ramp. The sweep measures Time to First Token (TTFT) degradation and the token-throughput saturation curve.

> The serving pillar **auto-sizes its own sample count** per config, so `--num-prompts` does not apply here (and is rejected with `--serving-only`); shape the workload with `--batch-sizes` / `--input-lengths` / `--output-lengths` instead.

**Serving target — two options (this notebook uses Ollama):**
* **Ollama (local, used below):** `--remote-endpoint http://localhost:11434/v1 --models gemma4-qat:4b --tokenizer google/gemma-4-E4B-it`.
* **vLLM (OpenAI-compatible remote):** first `vllm serve google/gemma-4-26B-A4B-it`, then swap the flags to `--remote-endpoint http://127.0.0.1:8000/v1 --models google/gemma-4-26B-A4B-it --tokenizer google/gemma-4-26B-A4B-it`.

> Note: the `unsloth/gemma-4-E4B-it-qat-GGUF` weights, `google/gemma-4-E4B-it` / `google/gemma-4-26B-A4B-it` tokenizer names, and the `google/gemma-4-26B-A4B-it` served id are the intended gemma-4 launch artifacts (placeholders until the model is public).

### Tuning the stress ramp to your hardware

Every serving cell below (concurrency, geometry, and the campaigns in §8) also runs the arrival-rate **stress ramp**: it finds the max request rate the server sustains while keeping up **and** meeting a latency SLO (default P99 TTFT ≤ 5000 ms, P99 inter-token ≤ 200 ms), over 3 sweep reps. On a modest laptop/Colab box the slow model can't gather 15 steady completions per rate, so every rate reads `TOO-FEW` and the knee is 0. For those boxes, relax the **sampling** (not the SLO):

| Flag | Controls | Default | Modest hardware |
|---|---|---|---|
| `--stress-min-samples <n>` | steady completions needed per rate point | `15` | `5` (registers a knee faster on a slow box) |
| `--stress-reps <n>` | sweeps per run (more = tighter CI, slower) | `3` | `1` |
| `--no-stress-test` | skip the ramp entirely | (on) | use when you only want serving latency |
| `--stress-threshold <ms>` | P99 **TTFT** SLO = what "sustainable" means | `5000` | keep near default |
| `--stress-tpot-threshold <ms>` | P99 **inter-token** SLO | `200` | keep near default |

**Keep the SLO sane — it *defines* the knee.** Raising `--stress-threshold` far (e.g. to 20000 ms) lets a hopelessly-queued rate "pass", so the reported knee balloons to ~2× the real capacity: it then measures queue depth, not sustainable performance. Relax the *sampling* knobs above for a slow box, not the SLO.

How it behaves on slow hardware: the sweep is **ascending** (probes low→high and stops at the first SLO miss), so it finds a small sustainable rate without ever flooding the server. If even a single request can't meet the TTFT SLO, a preflight **skips** the sweep with a "below stress floor" message — use `--no-stress-test` there (the serving latency numbers are the meaningful measurement).

In [ ]:
# Ollama (local) serving + stress sweep across 4 concurrency points.
# Serving auto-sizes its sample count, so we do NOT pass --num-prompts here.
# Modest-hardware stress config: one sweep + a lower steady-sample floor so a slow
# box registers a knee quickly. Keep the DEFAULT SLO - over-relaxing
# --stress-threshold would let a hopelessly-queued rate "pass" and inflate the knee.
!gbench --models gemma4-qat:4b \
        --batch-sizes 1 4 8 16 \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_concurrency \
        --num-iterations 1 --warmup-iterations 0 \
        --stress-reps 1 --stress-min-samples 5

# vLLM alternative (after `vllm serve google/gemma-4-26B-A4B-it`):
# !gbench --models google/gemma-4-26B-A4B-it \
#         --batch-sizes 1 4 8 16 \
#         --remote-endpoint http://127.0.0.1:8000/v1 \
#         --tokenizer google/gemma-4-26B-A4B-it \
#         --results-dir ./results_concurrency \
#         --num-iterations 1 --warmup-iterations 0 \
#         --stress-reps 1 --stress-min-samples 5

## 7b. Custom prompt geometry (input/output lengths)

Beyond concurrency, you can pin the exact prompt shape the serving pillar drives. `--input-lengths` sets the prompt (prefill) token count and `--output-lengths` sets the generated (decode) token count. Below we hold concurrency at a single point and force a decode-heavy geometry (short 256-token prompt, long 1024-token generation). The same flags work against either serving target.

In [ ]:
# Ollama (local): custom prompt geometry via --input-lengths / --output-lengths.
# Modest-hardware stress config (see the tuning table in section 7); default SLO kept.
!gbench --models gemma4-qat:4b \
        --batch-sizes 4 \
        --input-lengths 256 \
        --output-lengths 1024 \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_geometry \
        --num-iterations 1 --warmup-iterations 0 \
        --stress-reps 1 --stress-min-samples 5

# vLLM alternative (after `vllm serve google/gemma-4-26B-A4B-it`):
# !gbench --models google/gemma-4-26B-A4B-it \
#         --batch-sizes 4 \
#         --input-lengths 256 \
#         --output-lengths 1024 \
#         --remote-endpoint http://127.0.0.1:8000/v1 \
#         --tokenizer google/gemma-4-26B-A4B-it \
#         --results-dir ./results_geometry \
#         --num-iterations 1 --warmup-iterations 0 \
#         --stress-reps 1 --stress-min-samples 5

## 8. Standardized production workload campaigns

`gbench` includes 6 predefined workload campaigns that simulate production prompt geometries:
* **`chat-like`**: Multi-turn conversational traffic from the [ShareGPT V3 dataset](https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered).
* **`agentic`**: Long-context prompt ingestion (`8000` tokens) followed by a concise tool call response (`400` tokens).
* **`decode-heavy`**: Short prompt processing (`128` tokens) followed by high-volume generation (`2048` tokens).
* **`prefill-heavy`**: Long prompt ingestion (`8192` tokens) followed by short generation (`128` tokens).
* **`mixed`**: Balanced intermediate prompt processing (`4096` tokens / `1024` tokens).
* **`long-decode`**: High-context long-form generation (`8192` tokens / `8192` tokens).

We can inspect all built-in benchmark pillars via `!gbench --list pillars` or presets via `!gbench --list presets`, and then execute `agentic` and `decode-heavy` campaigns against our Ollama QAT model.

In [ ]:
# List available benchmark presets
!gbench --list presets

# Execute agentic and decode-heavy production campaigns (serving + stress by default).
# Each campaign pins its own input/output geometry; serving auto-sizes its sample
# count, so we do NOT pass --num-prompts here. Modest-hardware stress config
# (see the tuning table in section 7); default SLO kept.
!gbench --models gemma4-qat:4b \
        --campaign agentic \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_campaigns \
        --num-iterations 1 \
        --stress-reps 1 --stress-min-samples 5

!gbench --models gemma4-qat:4b \
        --campaign decode-heavy \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --results-dir ./results_campaigns \
        --num-iterations 1 \
        --stress-reps 1 --stress-min-samples 5

# vLLM alternative (after `vllm serve google/gemma-4-26B-A4B-it`): swap
#   --models google/gemma-4-26B-A4B-it
#   --remote-endpoint http://127.0.0.1:8000/v1
#   --tokenizer google/gemma-4-26B-A4B-it
# (keep the same --stress-* flags above)

## 9. Visualizing concurrency scaling curves

We use `matplotlib` to plot how Time to First Token (TTFT, P50) and output-token throughput scale as client concurrency (`batch_size`) grows from 1 to 16. Text and multimodal are drawn as **separate series** (each batch size runs both), so the vision-tower cost is directly comparable and the two don't get joined into one zig-zag line. Expect TTFT to climb with concurrency (steeply for MM) while throughput saturates.

In [ ]:
import json, glob, os
from pathlib import Path
import matplotlib.pyplot as plt

results_base = Path("./results_concurrency")
run_dirs = sorted([d for d in results_base.iterdir() if d.is_dir()], key=lambda d: d.stat().st_mtime, reverse=True) if results_base.exists() else []

if not run_dirs:
    print("No concurrency sweep directory found.")
else:
    latest_dir = run_dirs[0]
    summary_path = latest_dir / "summary.json"
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            rows = json.load(f).get("models", [])
    else:
        # Fallback: individual serving run JSONs (serving only by glob)
        rows = []
        for pf in sorted(latest_dir.glob("performance/serve_*.json")):
            with open(pf, "r", encoding="utf-8") as f:
                rows.append(json.load(f))

    # summary.json's "models" holds EVERY pillar; keep only SERVING rows (a stress
    # row has no batch_size/ttft). Each batch_size produces BOTH a text and an MM
    # serving row, so plot them as TWO SEPARATE series - drawing all rows as one
    # line sorted by batch_size connects text->MM at every x and yields a sawtooth.
    def _is_serving(r): return r.get("benchmark_type", "serving") in ("serving", "serving_multimodal")
    def _is_mm(r): return r.get("benchmark_type") == "serving_multimodal" or r.get("multimodal", False)
    # Median (P50) TTFT to match the CLI summary; throughput is output tokens/sec.
    def _ttft(r): return r.get("median_ttft_ms", r.get("mean_ttft_ms", r.get("ttft_p50_ms")))
    def _tput(r): return r.get("output_token_throughput", r.get("output_throughput", r.get("request_throughput")))

    serving = [r for r in rows if _is_serving(r) and r.get("batch_size") is not None]
    series = {
        "Text":       sorted([r for r in serving if not _is_mm(r)], key=lambda r: r["batch_size"]),
        "Multimodal": sorted([r for r in serving if _is_mm(r)],     key=lambda r: r["batch_size"]),
    }
    styles = {"Text": ("o", "tab:blue"), "Multimodal": ("s", "tab:orange")}

    if any(series.values()):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        for name, recs in series.items():
            if not recs:
                continue
            bs = [r["batch_size"] for r in recs]
            marker, color = styles[name]
            ax1.plot(bs, [_ttft(r) for r in recs], marker=marker, color=color, label=name)
            ax2.plot(bs, [_tput(r) for r in recs], marker=marker, color=color, label=name)
        ax1.set_title("TTFT (P50) vs Concurrency"); ax1.set_xlabel("Concurrent Batch Size")
        ax1.set_ylabel("TTFT (ms)"); ax1.grid(True); ax1.legend()
        ax2.set_title("Output Throughput vs Concurrency"); ax2.set_xlabel("Concurrent Batch Size")
        ax2.set_ylabel("Output Tokens / sec"); ax2.grid(True); ax2.legend()
        plt.tight_layout(); plt.show()
    else:
        print("No serving concurrency records found in:", latest_dir)


## 10. Session cleanup and server shutdown

We terminate the background Ollama process and delete temporary test Modelfiles.

In [ ]:
import subprocess, os

subprocess.run(["pkill", "-f", "ollama"], check=False)
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")